In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
hinlphmhng_checkpoints_path = kagglehub.dataset_download('hinlphmhng/checkpoints')
leehien12_uav_forest_sunny_path = kagglehub.dataset_download('leehien12/uav-forest-sunny')

print('Data source import complete.')


# NB03: Evaluate Old Checkpoints

**Mục đích**: Evaluate các checkpoint cũ (train bằng NB02 cũ, chưa gộp evaluate)
rồi ghi kết quả vào `results_all.csv` cùng format với NB04.

**Chạy**: Internet ON, GPU ON

**Input**:
- Dataset `checkpoints` (3 folders)
- Dataset `uav-forest-sunny-dataset-test` (seq9)

**Output**: `eval_results/results_all.csv` → input cho NB04

### Lưu ý quan trọng:
- Mỗi notebook cũ dùng **IMG_SIZE khác nhau** (512, 768, 1024)
- Checkpoint naming convention **khác nhau** giữa các notebook
- Checkpoint metadata format **khác nhau**

In [ ]:
# ============================================================
# CONFIG
# ============================================================
BATCH_SIZE  = 4
NUM_WORKERS = 4
USE_AMP     = True
TEST_SEQ    = 'seq9'

import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

CKPT_ROOT = Path('/kaggle/input/datasets/hinlphmhng/checkpoints') if IS_KAGGLE \
            else Path('checkpoints')
EVAL_DIR  = Path('/kaggle/working/eval_results') if IS_KAGGLE \
            else Path('outputs/eval_results')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# CHECKPOINT REGISTRY — khai báo thủ công cho từng experiment
#
# Lấy từ notebooks/Train/*.ipynb:
#   02-train-unetpp-resnet50.ipynb  → IMG_SIZE (512,512)
#   02-train-deeplabv3-mit-b5.ipynb → IMG_SIZE (768,768)
#   02-train-segformer-mit-b5.ipynb → IMG_SIZE (1024,1024)
# ============================================================

CHECKPOINT_REGISTRY = [
    {
        'folder':   '02_train_unetpp_resnet50',
        'model':    'unetpp',
        'encoder':  'resnet50',
        'seed':     42,
        'img_size': (512, 512),
        'ckpt_pattern': r'epoch_(\d+)_miou_([\d.]+)\.pth',
    },
    {
        'folder':   'deeplabv3_mit_b5',
        'model':    'deeplabv3plus',
        'encoder':  'mit_b5',
        'seed':     42,
        'img_size': (768, 768),
        # Naming: MIT-B5_e032_miou0.7128.pth
        'ckpt_pattern': r'MIT-B5_e(\d+)_miou([\d.]+)\.pth',
    },
    {
        'folder':   'segformer_mit_b5_seed42_checkpoints',
        'model':    'segformer',
        'encoder':  'mit_b5',
        'seed':     42,
        'img_size': (1024, 1024),
        'ckpt_pattern': r'epoch_(\d+)_miou_([\d.]+)\.pth',
    },
]

print(f'CKPT_ROOT: {CKPT_ROOT}')
print(f'EVAL_DIR:  {EVAL_DIR}')
print(f'Registered experiments: {len(CHECKPOINT_REGISTRY)}')
for r in CHECKPOINT_REGISTRY:
    print(f'  • {r["model"]}_{r["encoder"]} | img={r["img_size"]} | folder={r["folder"]}')

CKPT_ROOT: /kaggle/input/checkpoints
EVAL_DIR:  /kaggle/working/eval_results
Registered experiments: 3
  • unetpp_resnet50 | img=(512, 512) | folder=02_train_unetpp_resnet50
  • deeplabv3plus_mit_b5 | img=(768, 768) | folder=deeplabv3_mit_b5
  • segformer_mit_b5 | img=(1024, 1024) | folder=segformer_mit_b5_seed42_checkpoints


In [ ]:
import re, time, json, pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB


In [ ]:
# ============================================================
# DATASET CONSTANTS + ForestDataset
# ============================================================
LABEL_COLORS = [
    (0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
    (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0)
]
CLASS_NAMES = [
    'Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
    'Ground_vegetation','Rocks','Building','Fence','Car','Empty'
]
NUM_CLASSES = len(CLASS_NAMES)
PALETTE     = np.array(LABEL_COLORS, dtype=np.uint8)

COLOR_LUT = np.full((256, 256, 256), NUM_CLASSES - 1, dtype=np.int64)
for i, (r, g, b) in enumerate(LABEL_COLORS):
    COLOR_LUT[r, g, b] = i

def rgb_to_mask(rgb_np):
    return COLOR_LUT[rgb_np[:,:,0], rgb_np[:,:,1], rgb_np[:,:,2]]

def mask_to_rgb(mask):
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        out[mask == i] = PALETTE[i]
    return out

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples   = []
        root = Path(root)
        for seq in sequences:
            color_dir = root / seq / 'color'
            if not color_dir.exists():
                print(f'  [SKIP] {seq} — no color dir')
                continue
            for f in sorted(color_dir.glob('*.png')):
                lf = root / seq / 'labels' / f.name
                if lf.exists():
                    self.samples.append((f, lf))
        print(f'  Loaded: {len(self.samples)} samples from {sequences}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img_path, lbl_path = self.samples[i]
        img  = np.array(Image.open(img_path).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(lbl_path).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask.astype(np.int64))
            img, mask = t['image'], t['mask']
        return img.float(), mask.long()

print('Dataset class ready.')

Dataset class ready.


In [ ]:
# ============================================================
# FIND seq9 DATA
# ============================================================
if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(exist_ok=True)
    for dirpath, _, _ in os.walk('/kaggle/input/datasets/hinlphmhng/checkpoints', followlinks=True):
        if os.path.basename(dirpath) == 'color':
            seq_dir  = os.path.dirname(dirpath)
            seq_name = os.path.basename(seq_dir)
            if re.match(r'seq\d+$', seq_name):
                link = DATA_ROOT / seq_name
                if not link.exists():
                    os.symlink(seq_dir, link)
                    print(f'  Symlinked: {seq_name} -> {seq_dir}')
else:
    DATA_ROOT = Path('data/forest_sunny')

seq9_color = DATA_ROOT / TEST_SEQ / 'color'
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'seq9 exists: {seq9_color.exists()}')

if not seq9_color.exists():
    print('[WARN] seq9 not found via symlink, searching...')
    for p in Path('/kaggle/input').rglob('seq9/color'):
        print(f'  Found: {p.parent}')
        DATA_ROOT = p.parent.parent
        break

assert (DATA_ROOT / TEST_SEQ / 'color').exists(), 'seq9 NOT FOUND!'

DATA_ROOT: /kaggle/working/data
seq9 exists: True


In [ ]:
# ============================================================
# MODEL REGISTRY + HELPERS
# ============================================================
MODELS = {
    'unet':          lambda enc: smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'unetpp':        lambda enc: smp.UnetPlusPlus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda enc: smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'upernet':       lambda enc: smp.UPerNet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
    'segformer':     lambda enc: smp.Segformer(encoder_name=enc, encoder_weights=None, in_channels=3, classes=NUM_CLASSES),
}

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def calc_flops(model, img_size):
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.zeros(1, 3, *img_size).to(next(model.parameters()).device)
        fa = FlopCountAnalysis(model, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        return fa.total() / 1e9
    except Exception as e:
        print(f'  [FLOPs WARN] {e}')
        return -1.0

def calc_fps(model, img_size, n_runs=50):
    if DEVICE != 'cuda': return -1.0
    model.eval()
    dummy = torch.zeros(1, 3, *img_size).to(DEVICE)
    with torch.no_grad():
        for _ in range(10): model(dummy)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs): model(dummy)
    torch.cuda.synchronize()
    return n_runs / (time.time() - t0)

print('Model registry ready.')

Model registry ready.


In [ ]:
from pathlib import Path

CKPT_ROOT = Path('/kaggle/input/datasets/hinlphmhng/checkpoints')

print("=== LIST REAL FOLDERS ===")
for p in CKPT_ROOT.iterdir():
    print(f'[{p.name}]')

=== LIST REAL FOLDERS ===
[segformer_mit_b5_seed42_checkpoints]
[deeplabv3_mit_b5]
[02_train_unetpp_resnet50]


In [ ]:
# ============================================================
# SCAN CHECKPOINTS
# ============================================================
EXPERIMENTS = []

for reg in CHECKPOINT_REGISTRY:
    folder_path = CKPT_ROOT / reg['folder']
    print(f'\n--- {reg["folder"]} ---')

    if not folder_path.exists():
        print(f'  ❌ Folder not found: {folder_path}')
        continue

    # Scan tất cả .pth trong folder (kể cả subfolders)
    pattern = re.compile(reg['ckpt_pattern'])
    best_ckpt = None
    best_miou = -1

    for pth in sorted(folder_path.rglob('*.pth')):
        match = pattern.match(pth.name)
        if not match:
            print(f'  [SKIP] {pth.name} (not matching pattern)')
            continue

        epoch = int(match.group(1))
        miou  = float(match.group(2))
        print(f'  Found: {pth.name} | epoch={epoch} | val_mIoU={miou:.4f}')

        if miou > best_miou:
            best_miou = miou
            best_ckpt = pth

    if best_ckpt is None:
        print(f'  ❌ No matching checkpoints found!')
        # List what's actually in the folder
        pth_files = list(folder_path.rglob('*.pth'))
        if pth_files:
            print(f'  Files found: {[p.name for p in pth_files]}')
            print(f'  Expected pattern: {reg["ckpt_pattern"]}')
        continue

    config_nb04 = f'{reg["model"]}_{reg["encoder"]}'
    EXPERIMENTS.append({
        'config_nb04': config_nb04,
        'model':       reg['model'],
        'encoder':     reg['encoder'],
        'seed':        reg['seed'],
        'img_size':    reg['img_size'],
        'ckpt':        best_ckpt,
        'val_miou':    best_miou,
    })
    print(f'  ✅ Best: {best_ckpt.name} (mIoU={best_miou:.4f})')

print(f'\n{"="*60}')
print(f'Total: {len(EXPERIMENTS)} experiments to evaluate:')
for exp in EXPERIMENTS:
    print(f'  • {exp["config_nb04"]:30s} | img={exp["img_size"]} | {exp["ckpt"].name}')


--- 02_train_unetpp_resnet50 ---
  Found: epoch_004_miou_0.5219.pth | epoch=4 | val_mIoU=0.5219
  Found: epoch_005_miou_0.5328.pth | epoch=5 | val_mIoU=0.5328
  Found: epoch_007_miou_0.5456.pth | epoch=7 | val_mIoU=0.5456
  ✅ Best: epoch_007_miou_0.5456.pth (mIoU=0.5456)

--- deeplabv3_mit_b5 ---
  Found: MIT-B5_e001_miou0.0131.pth | epoch=1 | val_mIoU=0.0131
  Found: MIT-B5_e030_miou0.7048.pth | epoch=30 | val_mIoU=0.7048
  Found: MIT-B5_e031_miou0.7127.pth | epoch=31 | val_mIoU=0.7127
  Found: MIT-B5_e032_miou0.7128.pth | epoch=32 | val_mIoU=0.7128
  ✅ Best: MIT-B5_e032_miou0.7128.pth (mIoU=0.7128)

--- segformer_mit_b5_seed42_checkpoints ---
  Found: epoch_057_miou_0.6025.pth | epoch=57 | val_mIoU=0.6025
  Found: epoch_058_miou_0.6030.pth | epoch=58 | val_mIoU=0.6030
  ✅ Best: epoch_058_miou_0.6030.pth (mIoU=0.6030)

Total: 3 experiments to evaluate:
  • unetpp_resnet50                | img=(512, 512) | epoch_007_miou_0.5456.pth
  • deeplabv3plus_mit_b5           | img=(768, 768) |

In [ ]:
# ============================================================
# EVALUATE ALL CHECKPOINTS
# Mỗi model có thể dùng IMG_SIZE khác nhau!
# ============================================================
csv_path = EVAL_DIR / 'results_all.csv'

if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    print(f'Existing results: {len(df_existing)} rows')
else:
    df_existing = pd.DataFrame()
    print('No existing results — starting fresh')

all_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    config = exp['config_nb04']
    img_size = exp['img_size']

    # Skip nếu đã có
    if len(df_existing) > 0:
        already = df_existing[
            (df_existing['config'] == config) &
            (df_existing['seed'] == exp['seed'])
        ]
        if len(already) > 0:
            print(f'\n[{i}/{len(EXPERIMENTS)}] {config} — SKIP (already in CSV)')
            continue

    print(f'\n{"="*60}')
    print(f'[{i}/{len(EXPERIMENTS)}] {config}')
    print(f'  Checkpoint: {exp["ckpt"].name}')
    print(f'  IMG_SIZE:   {img_size}')

    try:
        # --- Build model & load weights ---
        model = MODELS[exp['model']](exp['encoder']).to(DEVICE)
        ckpt  = torch.load(exp['ckpt'], map_location=DEVICE, weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state)
        model.eval()
        print(f'  Model loaded ✅')

        # --- Efficiency metrics (dùng IMG_SIZE của model đó) ---
        params_M = count_params(model)
        flops_G  = calc_flops(model, img_size)
        fps      = calc_fps(model, img_size)
        print(f'  Params: {params_M:.2f}M | FLOPs: {flops_G:.2f}G | FPS: {fps:.1f}')

        # --- Build test loader VỚI IMG_SIZE ĐÚNG ---
        test_tf = A.Compose([
            A.Resize(*img_size),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2()
        ])
        test_ds = ForestDataset(DATA_ROOT, [TEST_SEQ], test_tf)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

        # --- Evaluate ---
        test_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
        saved_samples = []

        with torch.no_grad():
            for imgs, masks in tqdm(test_loader, desc=f'Testing {config}', leave=True):
                imgs = imgs.to(DEVICE)
                with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
                    out = model(imgs)
                preds = out.argmax(1).cpu().numpy()
                gts   = masks.numpy()

                for pred, gt in zip(preds, gts):
                    valid = (gt >= 0) & (gt < NUM_CLASSES)
                    np.add.at(test_cm, (gt[valid], pred[valid]), 1)

                if len(saved_samples) < 8:
                    for si in range(min(len(imgs), 8 - len(saved_samples))):
                        img_np = imgs[si].cpu().numpy().transpose(1,2,0)
                        mean_ = np.array([0.485, 0.456, 0.406])
                        std_  = np.array([0.229, 0.224, 0.225])
                        img_np = np.clip(img_np * std_ + mean_, 0, 1)
                        saved_samples.append((img_np, preds[si], gts[si]))

        # --- Compute metrics ---
        with np.errstate(divide='ignore', invalid='ignore'):
            t_inter = np.diag(test_cm)
            t_union = test_cm.sum(1) + test_cm.sum(0) - t_inter
            t_iou   = np.where(t_union > 0, t_inter / t_union, np.nan)
            test_miou = float(np.nanmean(t_iou))
            test_pa   = float(t_inter.sum() / (test_cm.sum() + 1e-9))

            t_prec = np.where(test_cm.sum(0) > 0, t_inter / test_cm.sum(0), np.nan)
            t_rec  = np.where(test_cm.sum(1) > 0, t_inter / test_cm.sum(1), np.nan)
            t_f1   = np.where((t_prec + t_rec) > 0, 2*t_prec*t_rec/(t_prec+t_rec), np.nan)
            test_mf1 = float(np.nanmean(t_f1))

        # --- Build result row (NB04 format) ---
        row = {
            'config':   config,
            'model':    exp['model'],
            'encoder':  exp['encoder'],
            'seed':     exp['seed'],
            'ckpt':     exp['ckpt'].name,
            'mIoU':     round(test_miou * 100, 2),
            'PA':       round(test_pa * 100, 2),
            'mF1':      round(test_mf1 * 100, 2),
            'params_M': round(params_M, 2),
            'flops_G':  round(flops_G, 2),
            'fps':      round(fps, 1),
        }
        for j, cls in enumerate(CLASS_NAMES):
            v = t_iou[j]
            row[f'iou_{cls}'] = round(float(v) * 100, 2) if not np.isnan(v) else -1.0

        all_results.append(row)

        # Print
        print(f'\n  ┌─── TEST RESULTS ───')
        print(f'  │ mIoU:   {row["mIoU"]}%')
        print(f'  │ PA:     {row["PA"]}%')
        print(f'  │ mF1:    {row["mF1"]}%')
        print(f'  │ Params: {row["params_M"]}M | FLOPs: {row["flops_G"]}G | FPS: {row["fps"]}')
        print(f'  └──────────────────')
        print(f'  Per-class IoU:')
        for j, cls in enumerate(CLASS_NAMES):
            v = t_iou[j]
            if not np.isnan(v):
                print(f'    {cls:20s}: {v*100:.2f}%')

        # Save artifacts
        np.save(EVAL_DIR / f'{config}_seed{exp["seed"]}_cm.npy', test_cm)
        with open(EVAL_DIR / f'{config}_seed{exp["seed"]}_samples.pkl', 'wb') as f:
            pickle.dump(saved_samples, f)

        # Auto-save CSV
        df_new = pd.DataFrame(all_results)
        df_all = pd.concat([df_existing, df_new], ignore_index=True) if len(df_existing) > 0 else df_new
        df_all.to_csv(csv_path, index=False)
        print(f'  CSV saved ({len(df_all)} total rows)')

        # Free GPU
        del model, ckpt, state, test_ds, test_loader
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'  ❌ ERROR: {e}')
        import traceback; traceback.print_exc()

# Final save
if all_results:
    df_new = pd.DataFrame(all_results)
    df_all = pd.concat([df_existing, df_new], ignore_index=True) if len(df_existing) > 0 else df_new
    df_all.to_csv(csv_path, index=False)
    print(f'\n{"="*60}')
    print(f'✅ DONE: {len(all_results)} new + {len(df_existing)} existing = {len(df_all)} total')
    print(f'   Saved: {csv_path}')
else:
    print('\nNo new results.')

No existing results — starting fresh

[1/3] unetpp_resnet50
  Checkpoint: epoch_007_miou_0.5456.pth
  IMG_SIZE:   (512, 512)
  Model loaded ✅
  Params: 48.99M | FLOPs: 230.47G | FPS: 12.5
  Loaded: 1397 samples from ['seq9']


Testing unetpp_resnet50: 100%|██████████| 350/350 [01:21<00:00,  4.31it/s]



  ┌─── TEST RESULTS ───
  │ mIoU:   60.51%
  │ PA:     86.83%
  │ mF1:    71.05%
  │ Params: 48.99M | FLOPs: 230.47G | FPS: 12.5
  └──────────────────
  Per-class IoU:
    Sky                 : 0.42%
    Deciduous_trees     : 66.29%
    Coniferous_trees    : 75.44%
    Fallen_trees        : 42.64%
    Dirt_ground         : 85.34%
    Ground_vegetation   : 81.84%
    Rocks               : 75.53%
    Building            : 87.75%
    Fence               : 34.43%
    Car                 : 55.39%
  CSV saved (1 total rows)

[2/3] deeplabv3plus_mit_b5
  Checkpoint: MIT-B5_e032_miou0.7128.pth
  IMG_SIZE:   (768, 768)
  Model loaded ✅
  Params: 82.60M | FLOPs: 224.91G | FPS: 4.2
  Loaded: 1397 samples from ['seq9']


Testing deeplabv3plus_mit_b5: 100%|██████████| 350/350 [03:04<00:00,  1.90it/s]



  ┌─── TEST RESULTS ───
  │ mIoU:   68.34%
  │ PA:     88.14%
  │ mF1:    78.18%
  │ Params: 82.6M | FLOPs: 224.91G | FPS: 4.2
  └──────────────────
  Per-class IoU:
    Sky                 : 10.11%
    Deciduous_trees     : 70.93%
    Coniferous_trees    : 77.37%
    Fallen_trees        : 50.09%
    Dirt_ground         : 87.25%
    Ground_vegetation   : 83.09%
    Rocks               : 78.72%
    Building            : 92.64%
    Fence               : 52.80%
    Car                 : 80.39%
  CSV saved (2 total rows)

[3/3] segformer_mit_b5
  Checkpoint: epoch_058_miou_0.6030.pth
  IMG_SIZE:   (1024, 1024)
  Model loaded ✅
  Params: 81.97M | FLOPs: 419.97G | FPS: 2.0
  Loaded: 1397 samples from ['seq9']


Testing segformer_mit_b5: 100%|██████████| 350/350 [06:23<00:00,  1.10s/it]



  ┌─── TEST RESULTS ───
  │ mIoU:   70.3%
  │ PA:     89.62%
  │ mF1:    78.52%
  │ Params: 81.97M | FLOPs: 419.97G | FPS: 2.0
  └──────────────────
  Per-class IoU:
    Sky                 : 0.82%
    Deciduous_trees     : 73.12%
    Coniferous_trees    : 79.94%
    Fallen_trees        : 56.79%
    Dirt_ground         : 89.76%
    Ground_vegetation   : 85.09%
    Rocks               : 83.57%
    Building            : 93.82%
    Fence               : 57.10%
    Car                 : 82.95%
  CSV saved (3 total rows)

✅ DONE: 3 new + 0 existing = 3 total
   Saved: /kaggle/working/eval_results/results_all.csv


In [ ]:
# ============================================================
# VIZ 1: Qualitative — best model
# ============================================================
if all_results:
    df_new = pd.DataFrame(all_results)
    best_row = df_new.sort_values('mIoU', ascending=False).iloc[0]
    best_config = best_row['config']
    best_seed   = int(best_row['seed'])

    viz_path = EVAL_DIR / f'{best_config}_seed{best_seed}_samples.pkl'
    if viz_path.exists():
        with open(viz_path, 'rb') as f:
            samples = pickle.load(f)

        n = min(6, len(samples))
        fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
        if n == 1: axes = axes[np.newaxis]

        for i in range(n):
            img, pred, gt = samples[i]
            axes[i, 0].imshow(img)
            axes[i, 0].set_title('Input', fontsize=11); axes[i, 0].axis('off')
            axes[i, 1].imshow(mask_to_rgb(gt))
            axes[i, 1].set_title('Ground Truth', fontsize=11); axes[i, 1].axis('off')
            axes[i, 2].imshow(mask_to_rgb(pred))
            axes[i, 2].set_title('Prediction', fontsize=11); axes[i, 2].axis('off')

        patches = [mpatches.Patch(color=np.array(LABEL_COLORS[ci])/255, label=CLASS_NAMES[ci])
                   for ci in range(NUM_CLASSES)]
        fig.legend(handles=patches, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.01),
                   fontsize=9, frameon=True)
        plt.suptitle(f'Best: {best_config} | mIoU={best_row["mIoU"]:.2f}%', fontsize=13, y=1.01)
        plt.tight_layout()
        plt.savefig(EVAL_DIR / 'figure_qualitative.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: figure_qualitative.png')
else:
    print('[SKIP]')

Saved: figure_qualitative.png


In [ ]:
# ============================================================
# VIZ 2: Confusion Matrices
# ============================================================
if all_results:
    n_models = len(all_results)
    fig, axes = plt.subplots(1, n_models, figsize=(12 * n_models, 10))
    if n_models == 1: axes = [axes]

    for idx, row in enumerate(all_results):
        cm_path = EVAL_DIR / f'{row["config"]}_seed{int(row["seed"])}_cm.npy'
        if cm_path.exists():
            cm = np.load(cm_path)
            cm_norm = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-9)
            sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                        ax=axes[idx], linewidths=0.5, vmin=0, vmax=1)
            axes[idx].set_xlabel('Predicted', fontsize=11)
            axes[idx].set_ylabel('Ground Truth', fontsize=11)
            axes[idx].set_title(f'{row["config"]}\nmIoU={row["mIoU"]}%', fontsize=12)
            axes[idx].tick_params(axis='x', rotation=40)

    plt.tight_layout()
    plt.savefig(EVAL_DIR / 'figure_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figure_confusion_matrices.png')
else:
    print('[SKIP]')

Saved: figure_confusion_matrices.png


In [ ]:
# ============================================================
# SUMMARY + NB04 COMPATIBILITY CHECK
# ============================================================
print('=' * 60)
print('NB03 XONG!')
print('=' * 60)

csv_path = EVAL_DIR / 'results_all.csv'
if csv_path.exists():
    df_final = pd.read_csv(csv_path)
    print(f'\nTotal: {len(df_final)} rows | {df_final["config"].nunique()} configs')
    print()
    cols = ['config', 'seed', 'mIoU', 'PA', 'mF1', 'params_M', 'flops_G', 'fps']
    avail = [c for c in cols if c in df_final.columns]
    print(df_final[avail].sort_values('mIoU', ascending=False).to_string(index=False))

    # NB04 GROUP_MAP check
    nb04_map = {
        'unet_resnet34': 'A', 'unet_resnet101': 'A',
        'deeplabv3plus_resnet101': 'A', 'unetpp_resnet50': 'A',
        'unet_efficientnet-b5': 'B', 'upernet_tu-convnext_base': 'B',
        'upernet_tu-convnext_large': 'B',
        'segformer_mit_b5': 'C',
        'upernet_tu-swinv2_base_window12to24_192to384': 'C',
        'upernet_tu-swinv2_large_window12to24_192to384': 'C',
        'unet_mit_b3': 'D', 'deeplabv3plus_mit_b3': 'D',
        'segformer_mit_b3': 'D', 'upernet_mit_b3': 'D',
    }
    print(f'\nNB04 GROUP_MAP:')
    for cfg in df_final['config'].unique():
        grp = nb04_map.get(cfg, '⚠️ NOT MAPPED')
        print(f'  {cfg:40s} -> {grp}')

print(f'\nOutput files:')
for p in sorted(EVAL_DIR.iterdir()):
    if p.is_file():
        sz = p.stat().st_size
        val = sz/1024 if sz < 1024**2 else sz/1024**2
        unit = 'KB' if sz < 1024**2 else 'MB'
        print(f'  {p.name:50s} {val:>7.1f} {unit}')

print(f'\n{"="*60}')
print('BƯỚC TIẾP:')
print('  1. Save Version trên Kaggle')
print('  2. Gộp CSV này với output NB02 mới (nếu có)')
print('  3. Add vào NB04 -> paper figures + LaTeX')
print(f'\n⚠️ LƯU Ý: deeplabv3plus_mit_b5 chưa có trong NB04 GROUP_MAP!')
print('   Thêm dòng này vào NB04 CONFIG cell:')
print("   'deeplabv3plus_mit_b5': 'C. Transformer',  # hoặc group phù hợp")

NB03 XONG!

Total: 3 rows | 3 configs

              config  seed  mIoU    PA   mF1  params_M  flops_G  fps
    segformer_mit_b5    42 70.30 89.62 78.52     81.97   419.97  2.0
deeplabv3plus_mit_b5    42 68.34 88.14 78.18     82.60   224.91  4.2
     unetpp_resnet50    42 60.51 86.83 71.05     48.99   230.47 12.5

NB04 GROUP_MAP:
  unetpp_resnet50                          -> A
  deeplabv3plus_mit_b5                     -> ⚠️ NOT MAPPED
  segformer_mit_b5                         -> C

Output files:
  deeplabv3plus_mit_b5_seed42_cm.npy                     1.1 KB
  deeplabv3plus_mit_b5_seed42_samples.pkl              180.0 MB
  figure_confusion_matrices.png                        280.7 KB
  figure_qualitative.png                                 6.8 MB
  results_all.csv                                        0.7 KB
  segformer_mit_b5_seed42_cm.npy                         1.1 KB
  segformer_mit_b5_seed42_samples.pkl                  320.0 MB
  unetpp_resnet50_seed42_cm.npy                  

In [ ]:
import shutil
import os

# Ensure the output directory exists and then zip its contents
if EVAL_DIR.exists():
    zip_filename = EVAL_DIR.parent / 'eval_results'
    shutil.make_archive(zip_filename, 'zip', EVAL_DIR)
    print(f'Đã nén tất cả file output vào: {zip_filename}.zip')
else:
    print(f'Thư mục output không tồn tại: {EVAL_DIR}')

# List the current working directory to show the created zip file
print('\nCác file trong thư mục làm việc hiện tại:')
for f in os.listdir('.'):
    print(f)

Đã nén tất cả file output vào: /kaggle/working/eval_results.zip

Các file trong thư mục làm việc hiện tại:
.virtual_documents
eval_results.zip
eval_results
data
